# EnPreNiNet — Google Colab GPU Runner (Pilot & E0–E5 Ablation Matrix)

Thin orchestration notebook for reproducing **EnPreNiNet** (Ranjitha & Atham, 2026) on BDD100K night-vision data (3 classes: `car`, `person`, `traffic light`).

**Pre-training Engineering Guarantees:**
- **Strict Run Isolation (`--run-id`):** Pilot (`pilot`) and Full Training (`full`) write into completely separate `runs/<run_id>/`, `weights/<run_id>/`, and `artifacts/<run_id>/` directories so Pilot can never overwrite Full Training outputs.
- **Fail-fast execution:** Every multi-step stage uses `%%bash` with `set -euo pipefail` so the pipeline stops immediately on any error.
- **Fast local SSD I/O:** Both `data/processed` (`/content/work/processed`) and YOLOv5 dataset views (`/content/work/yolo_views/`) live on local Colab SSD; only `weights`, `runs`, and `artifacts` are synced to Google Drive (`/content/drive/MyDrive/EnPreNiNet/`).
- **Pinned YOLOv5:** Locks `ultralytics/yolov5` to tag `v7.0` and logs both repo & YOLOv5 commit hashes to `artifacts/env_versions.txt`.

> **Runtime requirement:** Select `Runtime -> Change runtime type -> T4 / L4 / A100 GPU` before running.

## 1. Mount Google Drive, Clone Repository & Link Persistent Directories
- `weights/`, `runs/`, `artifacts/` -> Google Drive (`/content/drive/MyDrive/EnPreNiNet/`) for crash-safe persistence.
- `data/processed/` & `/content/work/yolo_views/` -> Local Colab SSD for high-throughput training I/O.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/drive/MyDrive/EnPreNiNet/weights /content/drive/MyDrive/EnPreNiNet/runs /content/drive/MyDrive/EnPreNiNet/artifacts
mkdir -p /content/work/processed /content/work/yolo_views
if [ ! -d /content/enpreninet-night-detection ]; then
  git clone https://github.com/thiemdv/enpreninet-night-detection.git /content/enpreninet-night-detection
else
  git -C /content/enpreninet-night-detection pull --ff-only
fi
cd /content/enpreninet-night-detection
mkdir -p data
rm -rf data/processed weights runs artifacts
ln -s /content/work/processed data/processed
ln -s /content/drive/MyDrive/EnPreNiNet/weights weights
ln -s /content/drive/MyDrive/EnPreNiNet/runs runs
ln -s /content/drive/MyDrive/EnPreNiNet/artifacts artifacts
ls -la data/processed weights runs artifacts

In [ ]:
%cd /content/enpreninet-night-detection

## 2. Install Dependencies (Preserve Colab CUDA PyTorch), Pin YOLOv5 (`v7.0`) & Log Environment

In [ ]:
%%bash
set -euo pipefail
grep -Ev '^[[:space:]]*(torch|torchvision)([[:space:]]|[<>=!~]|$)' requirements.txt > /tmp/enpreninet_reqs.txt
pip install -q -r /tmp/enpreninet_reqs.txt
if [ ! -d /content/yolov5 ]; then
  git clone https://github.com/ultralytics/yolov5.git /content/yolov5
fi
git -C /content/yolov5 fetch --tags
git -C /content/yolov5 checkout v7.0
grep -Ev '^[[:space:]]*(torch|torchvision)([[:space:]]|[<>=!~]|$)' /content/yolov5/requirements.txt > /tmp/yolov5_reqs.txt
pip install -q -r /tmp/yolov5_reqs.txt
nvidia-smi > artifacts/env_versions.txt
python --version >> artifacts/env_versions.txt
python -c "import torch, torchvision, cv2, numpy; print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'torchvision:', torchvision.__version__, 'cv2:', cv2.__version__, 'numpy:', numpy.__version__)" >> artifacts/env_versions.txt
echo "enpreninet_commit=$(git rev-parse HEAD)" >> artifacts/env_versions.txt
echo "yolov5_commit=$(git -C /content/yolov5 rev-parse HEAD)" >> artifacts/env_versions.txt
cat artifacts/env_versions.txt

## 3. Download & Prepare Raw BDD100K Dataset on Local SSD
**Option A (Recommended — Kaggle API, 0 MB Google Drive used):** Upload `kaggle.json` to `/content/drive/MyDrive/EnPreNiNet/kaggle.json`. This cell automatically pulls `solesensei/solesensei_bdd100k` directly to Colab local SSD (`/content/raw_bdd/`).

**Option B (Fallback — Google Drive ZIPs):** If `kaggle.json` is not found, unpacks `bdd100k_images_100k.zip` and `bdd100k_labels_release.zip` from `/content/drive/MyDrive/EnPreNiNet/raw_zips/`.

In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/raw_bdd
EXISTING_JSON=$(find /content/raw_bdd -type f -name 'bdd100k_labels_images_train.json' | head -n 1 || true)
if [ -z "${EXISTING_JSON}" ]; then
  if [ -f /content/drive/MyDrive/EnPreNiNet/kaggle.json ]; then
    echo 'Found kaggle.json on Drive -> downloading BDD100K from Kaggle directly to Colab SSD...'
    pip install -q kaggle
    mkdir -p ~/.kaggle
    cp /content/drive/MyDrive/EnPreNiNet/kaggle.json ~/.kaggle/kaggle.json
    chmod 600 ~/.kaggle/kaggle.json
    kaggle datasets download -d solesensei/solesensei_bdd100k -p /content/raw_bdd --unzip
  else
    echo 'No kaggle.json found -> unpacking BDD100K ZIPs from Google Drive...'
    unzip -q -o /content/drive/MyDrive/EnPreNiNet/raw_zips/bdd100k_images_100k.zip -d /content/raw_bdd/
    unzip -q -o /content/drive/MyDrive/EnPreNiNet/raw_zips/bdd100k_labels_release.zip -d /content/raw_bdd/
  fi
else
  echo "BDD100K already extracted at ${EXISTING_JSON} (skipping download)."
fi
TRAIN_IMG_DIR=$(find /content/raw_bdd -type d -path '*/images/100k/train' | head -n 1)
LABELS_JSON=$(find /content/raw_bdd -type f -name 'bdd100k_labels_images_train.json' | head -n 1)
test -n "${TRAIN_IMG_DIR}" && test -d "${TRAIN_IMG_DIR}"
test -n "${LABELS_JSON}" && test -f "${LABELS_JSON}"
RAW_IMAGES_ROOT=$(dirname "$(dirname "$(dirname "${TRAIN_IMG_DIR}")")")
printf 'RAW_IMAGES_ROOT="%s"\nLABELS_JSON="%s"\n' "${RAW_IMAGES_ROOT}" "${LABELS_JSON}" > /content/raw_bdd/bdd_paths.env
cat /content/raw_bdd/bdd_paths.env
echo "Train images count: $(find "${TRAIN_IMG_DIR}" -type f -name '*.jpg' | wc -l)"

## 4. Recommended GPU Pilot Experiment (200 Night Images, 2 Epochs — GHE Dual-Branch + E4/E5 Fusion)
Runs isolated under `--run-id pilot` (`runs/pilot/`, `weights/pilot/`, `artifacts/pilot/`) to validate real YOLOv5l `v7.0` runtime compatibility (`A07`), Faster R-CNN ResNet-152 GPU memory (`batch_size=2`), finite loss, E4 vs E5 fusion, and metrics before launching full E0–E5 training.

In [ ]:
%%bash
set -euo pipefail
source /content/raw_bdd/bdd_paths.env
python src/prepare_bdd.py --config configs/config.yaml --raw-dir "${RAW_IMAGES_ROOT}" --labels-json "${LABELS_JSON}" --out-dir /content/work/pilot_processed --max-images 200
python src/ghe_ensemble.py --config configs/config.yaml --run-id pilot --data-root /content/work/pilot_processed --method tiled_ghe_gray
python src/train_infer_yolo.py --config configs/config.yaml --run-id pilot --data-root /content/work/pilot_processed --work-dir /content/work/yolo_views/pilot --image-variant tiled_ghe_gray --yolo-repo /content/yolov5 --epochs 2 --batch-size 8 --device cuda
python src/train_infer_frcnn.py --config configs/config.yaml --run-id pilot --data-root /content/work/pilot_processed --image-variant tiled_ghe_gray --epochs 2 --batch-size 2 --device cuda
python src/fusion.py --config configs/config.yaml --run-id pilot --image-variant tiled_ghe_gray --no-global-nms --output runs/pilot/fusion_tiled_ghe_gray/fusion_tiled_ghe_gray_e4_noglobal_test_preds.pt --device cuda
python src/fusion.py --config configs/config.yaml --run-id pilot --image-variant tiled_ghe_gray --global-nms --output runs/pilot/fusion_tiled_ghe_gray/fusion_tiled_ghe_gray_e5_full_test_preds.pt --device cuda
python src/evaluate.py --config configs/config.yaml --run-id pilot --data-root /content/work/pilot_processed --eval-split test --preds Pilot_E1_YOLO=runs/pilot/yolo_tiled_ghe_gray/yolo_tiled_ghe_gray_test_preds.pt Pilot_E3_FRCNN=runs/pilot/frcnn_tiled_ghe_gray/frcnn_tiled_ghe_gray_test_preds.pt Pilot_E4_NoGlobal=runs/pilot/fusion_tiled_ghe_gray/fusion_tiled_ghe_gray_e4_noglobal_test_preds.pt Pilot_E5_Full=runs/pilot/fusion_tiled_ghe_gray/fusion_tiled_ghe_gray_e5_full_test_preds.pt
cat artifacts/pilot/ablation_table.md

## 5. Full Dataset Preparation (`prepare_bdd.py`) & GHE Ensemble (`ghe_ensemble.py`)

In [ ]:
%%bash
set -euo pipefail
source /content/raw_bdd/bdd_paths.env
python src/prepare_bdd.py --config configs/config.yaml --raw-dir "${RAW_IMAGES_ROOT}" --labels-json "${LABELS_JSON}"
python src/ghe_ensemble.py --config configs/config.yaml --run-id full --method tiled_ghe_gray
mkdir -p artifacts/full
cp data/processed/dataset_meta.json artifacts/full/dataset_meta.json
cp data/processed/images/tiled_ghe_gray/ghe_meta.json artifacts/full/ghe_meta_tiled_ghe_gray.json

In [ ]:
%%bash
set -euo pipefail
# Optional extended GHE mode ablation: color-preserving LAB, legacy sliding-window modes
python src/ghe_ensemble.py --config configs/config.yaml --run-id full --method tiled_ghe_lab
python src/ghe_ensemble.py --config configs/config.yaml --run-id full --method standard_local_he

## 6. Full Dual-Branch GPU Training & Inference (`E0`, `E1`, `E2`, `E3` — `--run-id full`)
- **E0:** Baseline YOLOv5l on `raw` night frames
- **E1:** GHE-YOLOv5l on `tiled_ghe_gray` frames
- **E2:** Baseline Faster R-CNN ResNet-152 on `raw` night frames
- **E3:** GHE-Faster R-CNN ResNet-152 on `tiled_ghe_gray` frames

In [ ]:
%%bash
set -euo pipefail
# E0: Baseline YOLOv5l (raw)
python src/train_infer_yolo.py --config configs/config.yaml --run-id full --work-dir /content/work/yolo_views/full --image-variant raw --yolo-repo /content/yolov5 --device cuda
# E1: GHE-YOLOv5l (tiled_ghe_gray)
python src/train_infer_yolo.py --config configs/config.yaml --run-id full --work-dir /content/work/yolo_views/full --image-variant tiled_ghe_gray --yolo-repo /content/yolov5 --device cuda

In [ ]:
%%bash
set -euo pipefail
# E2: Baseline Faster R-CNN ResNet-152 (raw) — AMP fp16 + auto-resume
python src/train_infer_frcnn.py --config configs/config.yaml --run-id full --image-variant raw --amp --resume --device cuda
# E3: GHE-Faster R-CNN ResNet-152 (tiled_ghe_gray) — AMP fp16 + auto-resume
python src/train_infer_frcnn.py --config configs/config.yaml --run-id full --image-variant tiled_ghe_gray --amp --resume --device cuda

## 7. EnPreNiNet Score-Level Fusion (`E4` vs `E5`) & Full Evaluation (`evaluate.py`)
- **E4:** Fusion without Global NMS (`--no-global-nms` ablation)
- **E5:** Full EnPreNiNet Fusion with class-wise Global NMS (`--global-nms`)

In [ ]:
%%bash
set -euo pipefail
# E4: EnPreNiNet Fusion without Global NMS
python src/fusion.py --config configs/config.yaml --run-id full --image-variant tiled_ghe_gray --no-global-nms --output runs/full/fusion_tiled_ghe_gray/fusion_tiled_ghe_gray_e4_noglobal_test_preds.pt --device cuda
# E5: EnPreNiNet Full Fusion with Global NMS
python src/fusion.py --config configs/config.yaml --run-id full --image-variant tiled_ghe_gray --global-nms --output runs/full/fusion_tiled_ghe_gray/fusion_tiled_ghe_gray_e5_full_test_preds.pt --device cuda
# Evaluate E0-E5 against identical test split & dataset fingerprint
python src/evaluate.py --config configs/config.yaml --run-id full --eval-split test --preds E0=runs/full/yolo_raw/yolo_raw_test_preds.pt E1=runs/full/yolo_tiled_ghe_gray/yolo_tiled_ghe_gray_test_preds.pt E2=runs/full/frcnn_raw/frcnn_raw_test_preds.pt E3=runs/full/frcnn_tiled_ghe_gray/frcnn_tiled_ghe_gray_test_preds.pt E4=runs/full/fusion_tiled_ghe_gray/fusion_tiled_ghe_gray_e4_noglobal_test_preds.pt E5=runs/full/fusion_tiled_ghe_gray/fusion_tiled_ghe_gray_e5_full_test_preds.pt
cat artifacts/full/ablation_table.md